In [ ]:
from pathlib import Path

import pandas as pd
import psycopg
from langchain_huggingface import HuggingFaceEmbeddings
from psycopg.rows import dict_row
import dotenv
import os

In [ ]:
from rds_chat_analysis.vector_db import setup_db, setup_indices
from rds_chat_analysis.vector_store_utils import (
    add_embeddings_to_db,
    build_vector_store_query,
)

## Connect to DB
These are the directories that contain your embeddings. See the `01-create-embeddings.ipynb` notebook for the expected structure and schema.

In [ ]:
MOCK_EMBEDDINGS_DIR = Path("./embeddings/mock")
PRIVATE_EMBEDDINGS_DIR = Path("./embeddings/private")

In [ ]:
# Connect to DB
dotenv.load_dotenv(dotenv_path=".env.mock", override=True)
dotenv.load_dotenv(dotenv_path=".env.private", override=True)


mock_db_settings = {
    "host": os.environ["POSTGRES_HOST_MOCK"],
    "port": os.environ.get("POSTGRES_PORT_MOCK", None),
    "dbname": os.environ["POSTGRES_DB_MOCK"],
    "user": os.environ["POSTGRES_USER_MOCK"],
    "password": os.environ["POSTGRES_PASSWORD_MOCK"],
    "sslmode": os.environ.get("DB_SSLMODE_MOCK", "require"),
}

private_db_settings = {
    "host": os.environ["POSTGRES_HOST_PRIVATE"],
    "port": os.environ.get("POSTGRES_PORT_PRIVATE", None),
    "dbname": os.environ["POSTGRES_DB_PRIVATE"],
    "user": os.environ["POSTGRES_USER_PRIVATE"],
    "password": os.environ["POSTGRES_PASSWORD_PRIVATE"],
    "sslmode": os.environ.get("DB_SSLMODE_PRIVATE", "require"),
}

# To prevent SSL connection errors
keepalive_kwargs = {
    "keepalives": 1,
    "keepalives_idle": 60,
    "keepalives_interval": 10,
    "keepalives_count": 5,
}

mock_conn = psycopg.connect(
    **mock_db_settings,
    row_factory=dict_row,
    **keepalive_kwargs,
)
private_conn = psycopg.connect(
    **private_db_settings,
    row_factory=dict_row,
    **keepalive_kwargs,
)

In [ ]:
from psycopg.pq.misc import connection_summary

print(f"Connected to Mock DB: {connection_summary(mock_conn.pgconn)}")
print(f"Connected to Private DB: {connection_summary(private_conn.pgconn)}")

## Setup DB schema

In [ ]:
# Set vector index settings
# NOTE: before using, be sure to allowlist `pg_diskann` and `vector` extensions in Azure

# DiskANN is only available in Azure flexible postgres, supports >> 5 million rows
# VECTOR_INDEX_SETTINGS = {
#     "index_type": "diskann",
#     "index_params": {
#         "max_neighbors": 32,
#         "l_value_ib": 100,
#     },
#     "distance_opclass": "vector_cosine_ops",
# }

# Alternative: HNSW for either local or Azure postgres, supports <= 5 million rows and has some inconsistencies with metadata post-filtering
VECTOR_INDEX_SETTINGS = {
    "index_type": "hnsw",
    "index_params": {"m": 16, "ef_construction": 64},
    "distance_opclass": "vector_cosine_ops",
}

In [ ]:
# Setup DB and schema
# - enable correct extensions (pgvector, pg_diskann)
# - create table for embeddings

setup_db(
    conn=mock_conn,
    table_name="log_embeddings",
    embedding_size=768,
    vector_index_settings=VECTOR_INDEX_SETTINGS,
    overwrite_existing=False,  # IMPORTANT: this will drop the table if it exists, and remove all your data.
)

setup_db(
    conn=private_conn,
    table_name="log_embeddings",
    embedding_size=768,
    vector_index_settings=VECTOR_INDEX_SETTINGS,
    overwrite_existing=False,  # IMPORTANT: this will drop the table if it exists, and remove all your data.
)

# Add data to DB

In [ ]:
# Add mock embeddings to mock DB
add_embeddings_to_db(
    conn=mock_conn,
    table_name="log_embeddings",
    embeddings_dir=MOCK_EMBEDDINGS_DIR,
    batch_size=64,
)

# Setup indices on mock DB (vector index + metadata index)
setup_indices(
    conn=mock_conn,
    table_name="log_embeddings",
    vector_index_settings=VECTOR_INDEX_SETTINGS,
    overwrite_existing=True,  # IMPORTANT: this will drop the index if it exists, and build a new one.
    pg_maintenance_mem="2GB",  # 4 workers * 2GB = 8GB RAM for building the vector index.
    pg_parallel_workers=4,
)

In [ ]:
private_conn.rollback()

In [ ]:
# Add private embeddings to private DB
add_embeddings_to_db(
    conn=private_conn,
    table_name="log_embeddings",
    embeddings_dir=PRIVATE_EMBEDDINGS_DIR,
    batch_size=64,
)

# Setup indices on private DB (vector index + metadata index)
# NOTE: for large datasets, building the vector index can take a long time. Please keep the notebook open until this cell is done.
setup_indices(
    conn=private_conn,
    table_name="log_embeddings",
    vector_index_settings=VECTOR_INDEX_SETTINGS,
    # overwrite_existing=True, # IMPORTANT: this will drop the index if it exists, and build a new one.
    # pg_maintenance_mem="2GB", # 4 workers * 2GB = 8GB RAM for building the vector index
    pg_parallel_workers=4,
)

# Test if everything works

In [ ]:
# Load embedder
import torch

cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")

# We use gte-multilingual, best mix of fast, small, high MTEB scores
MODEL_NAME = "Alibaba-NLP/gte-multilingual-base"

model_kwargs = {
    "device": "cuda:0" if cuda_available else "cpu",
    "trust_remote_code": True,  # Required to run gte-multilingual in sentence-transformers
}
encode_kwargs = {"normalize_embeddings": False}
embedder = HuggingFaceEmbeddings(
    model_name=MODEL_NAME, model_kwargs=model_kwargs, encode_kwargs=encode_kwargs
)

In [ ]:
# Create an example query
query, query_params = build_vector_store_query(
    query_embedding=embedder.embed_query("Messages about food and drinks"),
    table_name="log_embeddings",
    k=5,
    distance_threshold=0.5,
    filters={"role": "user", "language": "English"},
)

with private_conn.cursor() as cur:
    cur.execute(query, query_params)
    results = cur.fetchall()

pd.json_normalize(results)  # json_normalize flattens metadata to individual columns